In [8]:
# Load the extracted content CSV
import pandas as pd
df = pd.read_csv('iipcm_extracted_content.csv')
df.head()

,ark_url,title,date,creator,subject,description,item_type,source_url,full_text
0,https://digital.library.unt.edu/ark:/67531/met...,Restoring US First Website,2015-04-28,"AlSum, Ahmed (Stanford University)",digital preservation; web archiving; web crawl...,Presentation for the 2015 International Intern...,image_presentation,https://digital.library.unt.edu/ark:/67531/met...,IIPC GA 2015 – Stanford CA\nRestoring US First...
1,https://digital.library.unt.edu/ark:/67531/met...,Big UK Domain Data for the Arts and Humanities,2015-04-27,"Winters, Jane (University of London)",digital preservation; web archiving; big data;...,Presentation for the 2015 International Intern...,image_presentation,https://digital.library.unt.edu/ark:/67531/met...,Big UK Domain Data for the \nArts and Humaniti...
2,https://digital.library.unt.edu/ark:/67531/met...,Web Archive Information Retrieval,2015-04-28,"Costa, Miguel (Portuguese Web Archive); Gomes,...",digital preservation; web archiving; informati...,Presentation for the 2015 International Intern...,image_presentation,https://digital.library.unt.edu/ark:/67531/met...,Web Archive \nInformation Retrieval\nMiguel Co...
3,https://digital.library.unt.edu/ark:/67531/met...,WARCrefs for Deduplicating Web Archives,2015-04-28,"El Dakar, Youssef (Bibliotheca Alexandrina)",digital preservation; web archiving; web crawl...,Presentation for the 2015 International Intern...,image_presentation,https://digital.library.unt.edu/ark:/67531/met...,The BA web archive\n4 years of \nfocused web \...
4,https://digital.library.unt.edu/ark:/67531/met...,Reconstructing a Website's Lost Past,2015-04-28,"Nanni, Federico (University of Bologna)",digital preservation; web archiving; web crawl...,Presentation for the 2015 International Intern...,image_presentation,https://digital.library.unt.edu/ark:/67531/met...,History of universities\nScarcity of sources\n...


In [9]:
# Preprocessing loop using Groq Model Rotation & Gemini 3.1 Flash Lite fallback (Gemma fully removed)
import os
import time
import re
from tqdm import tqdm
from google.generativeai import GenerativeModel, configure as configure_gemini
from google.api_core.exceptions import ResourceExhausted as GeminiResourceExhausted
from groq import Groq, RateLimitError as GroqRateLimitError

# Force reload .env to override cached keys in Jupyter memory
try:
    from dotenv import load_dotenv
    if os.path.exists(r"../.env"):
        load_dotenv(dotenv_path=r"../.env", override=True)
except Exception:
    pass

gemini_api_key = os.getenv("GEMINI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

if not gemini_api_key:
    print("⚠️ WARNING: GEMINI_API_KEY environment variable not set. Please check .env")
if not groq_api_key or groq_api_key == "your_groq_api_key_here":
    print("⚠️ WARNING: GROQ_API_KEY environment variable not set correctly. Please check .env")

# Initialize API clients (Gemma removed completely)
configure_gemini(api_key=gemini_api_key)
gemini_model = GenerativeModel("gemini-3.1-flash-lite")

groq_client = Groq(api_key=groq_api_key)
GROQ_MODELS = [
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "llama-3.1-8b-instant"
]
current_groq_idx = 0

output_file = "../IIPC_cleaned_text.csv"
existing_df = None

# Load progress if file exists to support resumable runs
if os.path.exists(output_file):
    try:
        existing_df = pd.read_csv(output_file)
        print(f"📂 Found existing progress in {output_file}. Resuming from last run...")
    except Exception as e:
        print(f"⚠️ Failed to read existing {output_file}: {e}")

cleaned_texts = []
groq_daily_exhausted = False
prev_model_label = ""

# Process each row with rate-limit protection and checkpointing
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing with Model Router"):
    # Resume check
    if existing_df is not None and row["ark_url"] in existing_df["ark_url"].values:
        prev_val = existing_df.loc[existing_df["ark_url"] == row["ark_url"], "cleaned_text"].values[0]
        if pd.notna(prev_val) and str(prev_val).strip():
            cleaned_texts.append(prev_val)
            continue

    text = row["full_text"]
    if pd.isna(text) or not str(text).strip():
        cleaned_texts.append("")
        continue

    text_str = str(text)
    cleaned = ""
    
    # Route long docs or when both Groq models are daily-exhausted → Gemini
    current_route = "gemini_3.1" if (len(text_str) > 32000 or groq_daily_exhausted) else "groq"
    model_label = GROQ_MODELS[current_groq_idx] if current_route == "groq" else "gemini-3.1-flash-lite"
    if model_label != prev_model_label:
        tqdm.write(f"Model: {model_label}")
        prev_model_label = model_label

    attempt = 0
    max_attempts = 6
    gemini_retry_cooldown = 0

    while attempt < max_attempts:
        try:
            if current_route == "gemini_3.1":
                if gemini_retry_cooldown > 0:
                    print(f"\n⏳ Gemini cooldown {gemini_retry_cooldown:.0f}s remaining, sleeping...")
                    time.sleep(gemini_retry_cooldown)
                    gemini_retry_cooldown = 0
                response = gemini_model.generate_content(
                    f"Structure and clean the following conference presentation text. "
                    f"Preserve all specific metadata (such as author names, affiliations, conference titles) and technical content. "
                    f"Do not summarize the slides; convert them into clear, readable narrative markdown format. "
                    f"Avoid any introductory text (like 'Here is a restructured...') and output ONLY the restructured markdown:\n\n{text_str}"
                )
                if response.candidates and response.candidates[0].content.parts:
                    cleaned = response.candidates[0].content.parts[-1].text.strip()
                else:
                    cleaned = response.text.strip()
                break

            elif current_route == "groq":
                model_name = GROQ_MODELS[current_groq_idx]
                completion = groq_client.chat.completions.create(
                    model=model_name,
                    messages=[
                        {
                            "role": "user",
                            "content": f"Structure and clean the following conference presentation text. "
                                       f"Preserve all specific metadata (such as author names, affiliations, conference titles) and technical content. "
                                       f"Do not summarize the slides; convert them into clear, readable narrative markdown format. "
                                       f"Avoid any introductory text (like 'Here is a restructured...') and output ONLY the restructured markdown:\n\n{text_str}"
                        }
                    ]
                )
                cleaned = completion.choices[0].message.content.strip()
                break

        except GeminiResourceExhausted as e:
            attempt += 1
            if attempt >= max_attempts:
                print(f"\n⚠️ Gemini consistently exhausted after {max_attempts} attempts. Skipping row.")
                break
            backoff = min(15 * (2 ** (attempt - 1)), 180)
            print(f"\n⚠️ Gemini rate limited (attempt {attempt}/{max_attempts}). Backing off {backoff:.0f}s...")
            gemini_retry_cooldown = backoff
            time.sleep(backoff)

        except GroqRateLimitError as e:
            error_msg = str(e).lower()
            # Daily quota → rotate to next Groq model; if both exhausted, Gemini becomes primary for remaining rows
            if "daily" in error_msg or "tpd" in error_msg or "token" in error_msg:
                attempt += 1
                if current_groq_idx + 1 < len(GROQ_MODELS):
                    current_groq_idx += 1
                    print(f"\n🔄 Groq daily limit hit. Rotating to: {GROQ_MODELS[current_groq_idx]}")
                    current_route = "groq"
                else:
                    print("\n⚠️ All Groq models daily exhausted. Gemini is now primary for remaining rows.")
                    groq_daily_exhausted = True
                    current_route = "gemini_3.1"
                continue

            # RPM / minute limit → exponential backoff on same route
            attempt += 1
            sleep_time = 10.0
            try:
                if hasattr(e, 'response') and e.response is not None:
                    retry_after = e.response.headers.get("retry-after")
                    if retry_after:
                        sleep_time = float(retry_after)
            except Exception:
                pass
            sleep_time = min(sleep_time * (1.5 ** (attempt - 1)), 120)
            print(f"\n⚠️ [Groq] {GROQ_MODELS[current_groq_idx]} rate limit attempt {attempt}/{max_attempts}. Sleeping {sleep_time:.2f}s...")
            time.sleep(sleep_time + 1.0)

        except Exception as e:
            attempt += 1
            model_label = GROQ_MODELS[current_groq_idx] if current_route == "groq" else "gemini-3.1-flash-lite"
            print(f"\n⚠️ [{model_label}] Attempt {attempt}/{max_attempts} failed: {e}")
            # Any non-quota error on Groq → Gemini for this row only (don't rotate models)
            if current_route == "groq":
                print(f"🔄 {model_label} error. Falling back to Gemini for this row.")
                current_route = "gemini_3.1"
                continue
            # Already on Gemini → backoff and retry
            time.sleep(min(5 * attempt, 60))

    cleaned_texts.append(cleaned)
    
    # Save checkpoint on EVERY single item to guarantee zero progress is lost if paused/stopped
    full_cleaned_texts = cleaned_texts + [""] * (len(df) - len(cleaned_texts))
    df_checkpoint = df.copy()
    df_checkpoint["cleaned_text"] = full_cleaned_texts
    df_checkpoint.to_csv(output_file, index=False)

    # Safety sleeps: 2.1s for Groq (fast), 4.1s for Gemini 3.1 Flash Lite
    if current_route == "groq":
        time.sleep(2.1)
    else:
        time.sleep(4.1)

# Save final run
df["cleaned_text"] = cleaned_texts
df.to_csv(output_file, index=False)
print("✅ Processing complete. Saved to IIPC_cleaned_text.csv")

📂 Found existing progress in ../IIPC_cleaned_text.csv. Resuming from last run...


Processing with Model Router: 100%|██████████| 626/626 [00:00<00:00, 4502.49it/s]


✅ Processing complete. Saved to IIPC_cleaned_text.csv
